# Property-Rich Tree-Graph Walk Transformer (TGWT)
### Training & Benchmarking on Google Colab (NVIDIA Tesla T4 GPU)
Repository: [https://github.com/rajarajendra1103/TGWT.git](https://github.com/rajarajendra1103/TGWT.git)

This notebook trains and benchmarks **TGWT** against a parameter-matched standard causal Transformer baseline on **WikiText-103** across the 5 canonical comparison metrics:
1. **Perplexity (PPL) / Validation Loss**
2. **Throughput (Tokens / Second)**
3. **Peak VRAM / RAM Usage (MB)**
4. **FLOPs per Token**
5. **Zero-Shot / Next-Token Accuracy**

## Step 1: Verify NVIDIA Tesla T4 GPU
Ensure your Colab runtime is set to **GPU (T4)**: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi

## Step 2: Clone GitHub Repository & Install Requirements

In [ ]:
# Clone or pull latest code
import os
if not os.path.exists('/content/TGWT'):
    !git clone https://github.com/rajarajendra1103/TGWT.git /content/TGWT
    %cd /content/TGWT
else:
    %cd /content/TGWT
    !git fetch origin
    !git reset --hard origin/main

# Install dependencies
!pip install -q -r requirements.txt


## Step 3: Run Unit Tests
Verify all components (TreeBuilder, EdgeBuilder, Router, WalkSerializer, FusionLayer, Untouched Transformer, ScatterPoolingLayer, GatedUpdater) are functioning properly.

In [ ]:
!python -m unittest discover tests

## Step 4: Download and Prepare WikiText-103

In [ ]:
# Fast download & tokenization (50,000 samples in ~15 seconds with tqdm bar)
!python download_data.py --cache_dir ./data_cache --seq_len 1024 --max_train_samples 50000


## Step 5: Train TGWT on NVIDIA Tesla T4 GPU (FP16 Mixed Precision)

In [ ]:
!python train.py \
    --config configs/tgwt_colab_t4.yaml \
    --model_type tgwt \
    --max_steps 5000 \
    --batch_size 4 \
    --seq_len 1024 \
    --output_dir ./checkpoints_tgwt

## Step 6: Train Parameter-Matched Baseline Standard Transformer

In [ ]:
!python train.py \
    --config configs/baseline_colab_t4.yaml \
    --model_type baseline \
    --max_steps 5000 \
    --batch_size 4 \
    --seq_len 1024 \
    --output_dir ./checkpoints_baseline

## Step 7: Run the 5-Metric Head-to-Head Comparison Benchmark
This benchmark generates the comparison table showing how TGWT scales across context lengths (512, 1024, 2048, 4096).

In [ ]:
# Run extended 6-length scaling sweep (N = 512, 1024, 2048, 4096, 8192, 16384)
# Captures O(N^2) vs O(N*W) FLOPs and catches OOMs automatically
!python evaluate.py --test_scaling --batch_size 4
